# mPES — Colab GPU Retraining (DQN / RDQN / Transformer)

**Six cells. Set parameters → Runtime → Run all.**

Use this notebook to retrain `pes_dqn`, `pes_rdqn` or `pes_trf` from a previously
completed Bayesian optimisation, leveraging the Colab **GPU runtime** (cuDNN LSTM
for RDQN, GPU MatMul for DQN/TRF). On a T4/A100 the full `<PKG>_EPISODES` run
completes in minutes instead of hours.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or higher).

Workflow:
1. Edit the parameters in cell 1.
2. `Runtime → Run all`.
3. Trained model + plots are copied to Drive under
   `MyDrive/mPES/pes_<PKG>/<DATE>_<PKG>_TRAIN/` for download.

The notebook reads `best_params.json` either from the Git repo (default) or
from a previous on-Drive Bayesian-opt run via `BEST_PARAMS_DRIVE_DATE`.

In [ ]:
# ============================================================================
# Cell 1 - PARAMETERS (the only cell you normally edit)
# ============================================================================
# pylint: disable=missing-module-docstring
import os

PKG                     = 'rdqn'                    # dqn | rdqn | tr
NUM_EPISODES            = 0                         # 0 = use num_episodes from best_params.json; >0 overrides
BEST_PARAMS_DRIVE_DATE  = ''                        # 'YYYY-MM-DD' to pull best_params from Drive Bayesian-opt run; '' = use repo's inputs/best_params.json
GIT_REPO                = 'https://github.com/Maximiliano0/mPES_2026.git'
GIT_BRANCH              = 'drl_models'

os.environ['PKG']                     = PKG
os.environ['NUM_EPISODES']            = str(NUM_EPISODES)
os.environ['BEST_PARAMS_DRIVE_DATE']  = BEST_PARAMS_DRIVE_DATE
os.environ['GIT_REPO']                = GIT_REPO
os.environ['GIT_BRANCH']              = GIT_BRANCH
os.environ['MPES_USE_GPU']            = '1'

assert PKG in ('dqn', 'rdqn', 'tr'), f'PKG must be dqn|rdqn|tr, got {PKG!r}'
print(f'[CELL 1] PKG={PKG!r} NUM_EPISODES={NUM_EPISODES} BEST_PARAMS_DRIVE_DATE={BEST_PARAMS_DRIVE_DATE!r} branch={GIT_BRANCH!r}')


In [ ]:
# ============================================================================
# Cell 2 - MOUNT DRIVE + VERIFY GPU
# ============================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
import subprocess
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

try:
    _proc = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False)
    _ok = _proc.returncode == 0 and 'GPU 0' in _proc.stdout
    _msg = _proc.stdout.strip() if _ok else (_proc.stdout + _proc.stderr)
except FileNotFoundError:
    _ok, _msg = False, 'nvidia-smi not installed (CPU runtime)'
if not _ok:
    raise RuntimeError(
        'No GPU detected (' + _msg.strip() + '). '
        'Switch the runtime: Runtime -> Change runtime type -> T4 GPU, '
        'then Runtime -> Run all again.'
    )
print('[OK]', _msg)


In [ ]:
# ============================================================================
# Cell 3 - CLONE/UPDATE REPO + INSTALL DEPS
# ============================================================================
import os, subprocess

_script = r'''
set -euo pipefail
REPO_DIR=/content/Win_mPES
if [[ ! -d "$REPO_DIR/.git" ]]; then
    git clone --branch "$GIT_BRANCH" "$GIT_REPO" "$REPO_DIR"
else
    cd "$REPO_DIR" && git fetch --all && git checkout "$GIT_BRANCH" && git pull
fi
cd "$REPO_DIR"
bash utils/colab/setup_colab.sh
'''
subprocess.run(_script, shell=True, executable='/bin/bash', check=True, env=os.environ.copy())

In [ ]:
# ============================================================================
# Cell 4 - LOAD best_params.json (from Drive if BEST_PARAMS_DRIVE_DATE set)
# ============================================================================
import os, shutil

PKG = os.environ['PKG']
PKG_DIR_NAME = {'tr': 'pes_trf'}.get(PKG, f'pes_{PKG}')
REPO_INPUTS  = f'/content/Win_mPES/{PKG_DIR_NAME}/inputs'
REPO_BP      = os.path.join(REPO_INPUTS, 'best_params.json')

drive_date = os.environ.get('BEST_PARAMS_DRIVE_DATE', '').strip()
if drive_date:
    drive_bp = (
        f'/content/drive/MyDrive/mPES/{PKG_DIR_NAME}/'
        f'{drive_date}_BAYESIAN_OPT/best_params_{drive_date}.json'
    )
    if not os.path.isfile(drive_bp):
        raise FileNotFoundError(f'Drive best_params not found: {drive_bp}')
    os.makedirs(REPO_INPUTS, exist_ok=True)
    shutil.copy2(drive_bp, REPO_BP)
    print(f'[CELL 4] Copied {drive_bp}\n            -> {REPO_BP}')
else:
    if not os.path.isfile(REPO_BP):
        raise FileNotFoundError(
            f'No best_params.json in repo at {REPO_BP}. '
            'Either commit one to the branch or set BEST_PARAMS_DRIVE_DATE in cell 1.'
        )
    print(f'[CELL 4] Using repo best_params: {REPO_BP}')

import json
with open(REPO_BP, 'r', encoding='utf-8') as _f:
    _bp = json.load(_f)
print(f'   best trial #{_bp.get("best_trial_number")}  mean_perf={_bp.get("mean_perf"):.4f}  seed={_bp.get("trial_seed")}')

In [ ]:
# ============================================================================
# Cell 5 - RUN TRAINING (foreground, GPU, live progress)
# ============================================================================
# pylint: disable=reimported
import json
import os
import subprocess
import sys
import time

PKG = os.environ['PKG']
PKG_DIR_NAME = {'tr': 'pes_trf'}.get(PKG, f'pes_{PKG}')
REPO_BP = f'/content/Win_mPES/{PKG_DIR_NAME}/inputs/best_params.json'

MODULE = {
    'dqn':  'pes_dqn.ext.train_dqn',
    'rdqn': 'pes_rdqn.ext.train_rdqn',
    'tr':   'pes_trf.ext.train_transformer',
}[PKG]

# Resolve num_episodes: explicit override > best_params.json value.
_n_override = int(os.environ.get('NUM_EPISODES', '0'))
if _n_override > 0:
    _num_episodes = _n_override
    _episodes_src = f'CLI override ({_n_override})'
else:
    with open(REPO_BP, 'r', encoding='utf-8') as _f:
        _bp = json.load(_f)
    _num_episodes = int(_bp['hyperparameters']['num_episodes'])
    _episodes_src = f'best_params.json (trial #{_bp.get("best_trial_number")})'

print(f'[CELL 5] episodes={_num_episodes}  source={_episodes_src}')

_script = f"""
set -uo pipefail
cd /content/Win_mPES
source /content/mpes_env.sh
python -u -m {MODULE} {_num_episodes}
"""

env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
env['PYTHONUNBUFFERED'] = '1'

# Stream output line-by-line so progress is visible in real time.
# Each `Episode N Average Reward: ...` line gets a wall-clock timestamp + ETA.
_t0 = time.time()
_last_ep = 0
_proc = subprocess.Popen(
    _script, shell=True, executable='/bin/bash',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env,
)
assert _proc.stdout is not None
for _line in _proc.stdout:
    _line = _line.rstrip()
    if _line.startswith('Episode ') and 'Average Reward' in _line:
        try:
            _ep = int(_line.split()[1])
        except (ValueError, IndexError):
            _ep = 0
        _elapsed = time.time() - _t0
        _pct = 100.0 * _ep / _num_episodes if _num_episodes else 0.0
        _eta = (_elapsed / _ep) * (_num_episodes - _ep) if _ep else 0.0
        print(f'[{_elapsed/60:6.1f} min | {_pct:5.1f}% | ETA {_eta/60:5.1f} min] {_line}', flush=True)
        _last_ep = _ep
    else:
        print(_line, flush=True)
_proc.wait()
if _proc.returncode != 0:
    raise RuntimeError(f'training exited with code {_proc.returncode}')
print(f'\n[CELL 5] Done in {(time.time()-_t0)/60:.1f} min  (last reported episode: {_last_ep})')


In [ ]:
# ============================================================================
# Cell 6 - COPY TRAINED ARTIFACTS TO DRIVE
# ============================================================================
import os, glob, shutil

PKG = os.environ['PKG']
PKG_DIR_NAME = {'tr': 'pes_trf'}.get(PKG, f'pes_{PKG}')
TRAIN_TAG    = {'dqn': 'DQN_TRAIN', 'rdqn': 'RDQN_TRAIN', 'tr': 'TRF_TRAIN'}[PKG]

REPO_INPUTS  = f'/content/Win_mPES/{PKG_DIR_NAME}/inputs'
DRIVE_INPUTS = f'/content/drive/MyDrive/mPES/{PKG_DIR_NAME}'
os.makedirs(DRIVE_INPUTS, exist_ok=True)

# Find the most-recent <DATE>_<PKG>_TRAIN/ produced by this run.
_runs = sorted(glob.glob(os.path.join(REPO_INPUTS, f'*_{TRAIN_TAG}')))
if not _runs:
    raise FileNotFoundError(f'No *_{TRAIN_TAG} directory found under {REPO_INPUTS}')
_latest = _runs[-1]
_dst = os.path.join(DRIVE_INPUTS, os.path.basename(_latest))
if os.path.exists(_dst):
    shutil.rmtree(_dst)
shutil.copytree(_latest, _dst)
print(f'[CELL 6] Copied {_latest}\n            -> {_dst}')

# Also mirror the canonical (overwrite-on-each-run) artifacts at inputs/ root.
for _name in os.listdir(REPO_INPUTS):
    _src = os.path.join(REPO_INPUTS, _name)
    if os.path.isfile(_src) and (_name.endswith('.keras') or _name.endswith('.npy')):
        shutil.copy2(_src, os.path.join(DRIVE_INPUTS, _name))
        print(f'   mirror -> {os.path.join(DRIVE_INPUTS, _name)}')

print('\nDone. Download from Drive or rerun monitor notebook for inspection.')